In [2]:
import pandas as pd

df = pd.read_csv('/content/menstrual_cycle_dataset_with_factors.csv')
display(df.head())

,User ID,Age,BMI,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms
0,1,18,29.28,2,Moderate,5.4,Low Carb,2024-11-13 20:52:34.915012,26,7,2024-12-09 20:52:34.915012,Headache
1,1,18,29.28,2,Moderate,5.4,Low Carb,2024-12-09 20:52:34.915012,32,5,2025-01-10 20:52:34.915012,Fatigue
2,1,18,29.28,2,Moderate,5.4,Low Carb,2025-01-10 20:52:34.915012,41,7,2025-02-20 20:52:34.915012,Fatigue
3,1,18,29.28,2,Moderate,5.4,Low Carb,2025-02-20 20:52:34.915012,27,3,2025-03-19 20:52:34.915012,Fatigue
4,1,18,29.28,2,Moderate,5.4,Low Carb,2025-03-19 20:52:34.915012,42,5,2025-04-30 20:52:34.915012,Cramps


### 1. Data Cleaning & Temporal Alignment

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Convert date columns to datetime objects
df['Cycle Start Date'] = pd.to_datetime(df['Cycle Start Date'])
df['Next Cycle Start Date'] = pd.to_datetime(df['Next Cycle Start Date'])

# Sort the dataset chronologically per user
df = df.sort_values(by=['User ID', 'Cycle Start Date']).reset_index(drop=True)

print("Date columns converted and dataset sorted by User ID and Cycle Start Date.")
display(df.head())

Date columns converted and dataset sorted by User ID and Cycle Start Date.


,User ID,Age,BMI,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms
0,1,18,29.28,2,Moderate,5.4,Low Carb,2024-11-13 20:52:34.915012,26,7,2024-12-09 20:52:34.915012,Headache
1,1,18,29.28,2,Moderate,5.4,Low Carb,2024-12-09 20:52:34.915012,32,5,2025-01-10 20:52:34.915012,Fatigue
2,1,18,29.28,2,Moderate,5.4,Low Carb,2025-01-10 20:52:34.915012,41,7,2025-02-20 20:52:34.915012,Fatigue
3,1,18,29.28,2,Moderate,5.4,Low Carb,2025-02-20 20:52:34.915012,27,3,2025-03-19 20:52:34.915012,Fatigue
4,1,18,29.28,2,Moderate,5.4,Low Carb,2025-03-19 20:52:34.915012,42,5,2025-04-30 20:52:34.915012,Cramps


### 2. Feature Engineering (Biological & Lagged Memory)

In [4]:
# Calculate 'Cycle Length' for each cycle
df['Cycle Length'] = (df['Next Cycle Start Date'] - df['Cycle Start Date']).dt.days

# Group by 'User ID' for user-specific feature engineering
df_fe = df.groupby('User ID').apply(lambda x: x.assign(
    # Lagged cycle lengths
    prev_cycle_len_1=x['Cycle Length'].shift(1),
    prev_cycle_len_2=x['Cycle Length'].shift(2),
    # Lagged period length
    prev_period_len_1=x['Period Length'].shift(1),
    # Expanding/cumulative mean and std of cycle lengths
    user_running_mean=x['Cycle Length'].expanding().mean().shift(1),
    user_running_std=x['Cycle Length'].expanding().std().shift(1)
)).reset_index(drop=True)

# Fill early missing std values with 0 as requested
df_fe['user_running_std'] = df_fe['user_running_std'].fillna(0)

# Extract 'start_month'
df_fe['start_month'] = df_fe['Cycle Start Date'].dt.month

print("Feature engineering complete. New features added:")
display(df_fe.head())

Feature engineering complete. New features added:


/tmp/ipykernel_792/2019845277.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_fe = df.groupby('User ID').apply(lambda x: x.assign(


,User ID,Age,BMI,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,prev_cycle_len_1,prev_cycle_len_2,prev_period_len_1,user_running_mean,user_running_std,start_month
0,1,18,29.28,2,Moderate,5.4,Low Carb,2024-11-13 20:52:34.915012,26,7,2024-12-09 20:52:34.915012,Headache,NaN,NaN,NaN,NaN,0.000000,11
1,1,18,29.28,2,Moderate,5.4,Low Carb,2024-12-09 20:52:34.915012,32,5,2025-01-10 20:52:34.915012,Fatigue,26.0,NaN,7.0,26.0,0.000000,12
2,1,18,29.28,2,Moderate,5.4,Low Carb,2025-01-10 20:52:34.915012,41,7,2025-02-20 20:52:34.915012,Fatigue,32.0,26.0,5.0,29.0,4.242641,1
3,1,18,29.28,2,Moderate,5.4,Low Carb,2025-02-20 20:52:34.915012,27,3,2025-03-19 20:52:34.915012,Fatigue,41.0,32.0,7.0,33.0,7.549834,2
4,1,18,29.28,2,Moderate,5.4,Low Carb,2025-03-19 20:52:34.915012,42,5,2025-04-30 20:52:34.915012,Cramps,27.0,41.0,3.0,31.5,6.855655,3


### 3. Leakage Prevention & Dimensional Cleansing

In [5]:
# Explicitly drop leaky and redundant columns
columns_to_drop = [
    'Cycle Start Date',
    'Next Cycle Start Date',
    'Period Length', # This is now 'prev_period_len_1'
    'User ID' # Drop temporarily for one-hot encoding, will keep for LSTM sequence generation
]

# Make a copy to avoid SettingWithCopyWarning
df_clean = df_fe.copy()

df_clean = df_clean.drop(columns=columns_to_drop)

# Handle categorical string features using one-hot encoding
categorical_cols = ['Stress Level', 'Exercise Frequency', 'Diet', 'Symptoms']
# Ensure 'Stress Level' is treated as a numerical category if it represents ordinal values like 1, 2, 3.
# For now, treating it as categorical for demonstration of get_dummies.

# Convert 'Stress Level' to string first if it's currently numeric, to ensure it's one-hot encoded
# if the intent is to treat it as distinct categories rather than ordinal numbers.
# If 'Stress Level' is truly ordinal (e.g., 1=low, 5=high), it might be better to leave it as numeric
# or use ordinal encoding. For this task, assuming get_dummies is safe for all specified 'string features'.

for col in categorical_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str) # Ensure they are string types


df_clean = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, dtype=float) # drop_first to avoid multicollinearity

# Drop all rows containing NaN values generated during the lag shifting step
original_rows = len(df_clean)
df_clean.dropna(inplace=True)
final_rows = len(df_clean)

print(f"Dropped {original_rows - final_rows} rows due to NaN values after feature engineering.")
print("Data after leakage prevention and cleansing:")
display(df_clean.head())
print(f"Shape of cleaned data: {df_clean.shape}")

Dropped 200 rows due to NaN values after feature engineering.
Data after leakage prevention and cleansing:


,Age,BMI,Sleep Hours,Cycle Length,prev_cycle_len_1,prev_cycle_len_2,prev_period_len_1,user_running_mean,user_running_std,start_month,...,Stress Level_5,Exercise Frequency_Low,Exercise Frequency_Moderate,Diet_High Sugar,Diet_Low Carb,Diet_Vegetarian,Symptoms_Cramps,Symptoms_Fatigue,Symptoms_Headache,Symptoms_Mood Swings
2,18,29.28,5.4,41,32.0,26.0,5.0,29.000000,4.242641,1,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,18,29.28,5.4,27,41.0,32.0,7.0,33.000000,7.549834,2,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,18,29.28,5.4,42,27.0,41.0,3.0,31.500000,6.855655,3,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
5,18,29.28,5.4,41,42.0,27.0,5.0,33.600000,7.569676,4,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
6,18,29.28,5.4,31,41.0,42.0,5.0,34.833333,7.413951,6,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


Shape of cleaned data: (695, 23)


### 5. Target Scaling Evaluation Check (for Tree Models)

In [6]:
# Log-transform the target cycle length variable using np.log1p
# This transformation is typically applied during the training phase for tree models
# to help cope with sudden, irregular cycle spikes.

# First, define X and y from the cleaned dataframe
X = df_clean.drop(columns=['Cycle Length'])
y = df_clean['Cycle Length']

# Apply log1p transformation to the target variable
y_log_transformed = np.log1p(y)

print("Original target (first 5 values):")
print(y.head())
print("\nLog-transformed target (first 5 values) using np.log1p:")
print(y_log_transformed.head())

# Note: This transformed target `y_log_transformed` would be used for training
# tree-based models like Random Forest or XGBoost. When making predictions,
# remember to inverse transform (np.expm1) the predictions to get original scale values.

Original target (first 5 values):
2    41
3    27
4    42
5    41
6    31
Name: Cycle Length, dtype: int64

Log-transformed target (first 5 values) using np.log1p:
2    3.737670
3    3.332205
4    3.761200
5    3.737670
6    3.465736
Name: Cycle Length, dtype: float64


### 4. Split and Shape for Models - Format A (For XGBoost/Random Forest)

In [7]:
# Ensure all boolean fields from get_dummies are cast to numeric floats
# pd.get_dummies with dtype=float already handles this, but an explicit check can be added if needed.

# Define features (X) and target (y)
X = df_clean.drop(columns=['Cycle Length'])
y = df_clean['Cycle Length'] # Using original target for split, log-transform later for specific models

# Perform train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df_fe.loc[y.index, 'User ID'] if 'User ID' in df_fe.columns else None)

# Note: Stratification by 'User ID' ensures that each user's data is proportionally represented in train and test sets.
# However, `df_fe.loc[y.index, 'User ID']` is needed because `y` has dropped NaNs and its index might not align with `df_fe`'s original index. If `df_clean` doesn't retain original `User ID` this stratification might need adjustment.
# For now, let's proceed without `stratify` if 'User ID' is not directly available in `y`'s index.

# Re-doing train_test_split without stratify for simplicity given `User ID` was dropped from `df_clean`.
# If user-specific splitting is critical, the `df_fe` (before dropping User ID) should be used for splitting indices.

# Let's re-evaluate how to handle `User ID` for splitting
# A more robust approach for multi-user datasets is to split by user first, then within user, or use GroupShuffleSplit.

# For the sake of providing a working example, let's do a simple split without stratification by User ID
# if 'User ID' is not readily available in the final `df_clean`'s index.
# Assuming the previous drop of 'User ID' from `df_clean` means we can't stratify by it directly with `df_clean`'s index.

# To correctly stratify by User ID or ensure user-specific split:
# 1. Keep 'User ID' in `df_fe` until splitting.
# 2. Split `df_fe` into train/test indices first, then apply to X and y.

# Let's adjust by not dropping 'User ID' from df_fe until the very end, and use it for splitting.
# Rerunning from Feature Engineering to keep User ID until splitting

# Re-executing earlier steps to retain 'User ID' for splitting purposes
# This code block assumes df_fe is available from previous steps and still contains 'User ID'

df_with_user_id = df_fe.copy()

# Drop 'Cycle Start Date', 'Next Cycle Start Date', and 'Period Length'
columns_to_drop_for_features = [
    'Cycle Start Date',
    'Next Cycle Start Date',
    'Period Length' # This is now 'prev_period_len_1'
]
df_with_user_id = df_with_user_id.drop(columns=columns_to_drop_for_features)

# Handle categorical string features using one-hot encoding
categorical_cols = ['Stress Level', 'Exercise Frequency', 'Diet', 'Symptoms']
for col in categorical_cols:
    if col in df_with_user_id.columns:
        df_with_user_id[col] = df_with_user_id[col].astype(str)
df_final_features = pd.get_dummies(df_with_user_id, columns=categorical_cols, drop_first=True, dtype=float)

# Drop all rows containing NaN values
df_final_features.dropna(inplace=True)

# Now, define X, y, and the user_ids for splitting
X = df_final_features.drop(columns=['Cycle Length', 'User ID'])
y = df_final_features['Cycle Length']
user_ids_for_split = df_final_features['User ID']

# Ensure `user_ids_for_split` has the same index as `X` and `y`
assert X.index.equals(user_ids_for_split.index)
assert y.index.equals(user_ids_for_split.index)

# Perform train_test_split, stratifying by User ID
# We need to ensure that the User IDs passed to stratify are compatible with the indices of X and y.
# For stratification to work with groups, you generally want a 'group' column, not necessarily a target.
# However, `stratify` argument works on the target or a similar 1D array.
# A better approach for multi-user data is `GroupShuffleSplit` if full group separation is desired.

# For basic train_test_split with user distribution, a simpler approach:
# Get unique user IDs
unique_users = user_ids_for_split.unique()

# Split users into train and test sets
np.random.seed(42) # for reproducibility
train_users, test_users = train_test_split(unique_users, test_size=0.2, random_state=42)

# Create train and test datasets based on user IDs
X_train = X[user_ids_for_split.isin(train_users)]
X_test = X[user_ids_for_split.isin(test_users)]
y_train = y[user_ids_for_split.isin(train_users)]
y_test = y[user_ids_for_split.isin(test_users)]

# Cast all boolean fields to numeric floats (get_dummies already handles this if dtype=float)
# X_train = X_train.astype(float)
# X_test = X_test.astype(float)

print("Data split for XGBoost/Random Forest (Format A):")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

display(X_train.head())

Data split for XGBoost/Random Forest (Format A):
X_train shape: (546, 22), y_train shape: (546,)
X_test shape: (149, 22), y_test shape: (149,)


,Age,BMI,Sleep Hours,prev_cycle_len_1,prev_cycle_len_2,prev_period_len_1,user_running_mean,user_running_std,start_month,Stress Level_2,...,Stress Level_5,Exercise Frequency_Low,Exercise Frequency_Moderate,Diet_High Sugar,Diet_Low Carb,Diet_Vegetarian,Symptoms_Cramps,Symptoms_Fatigue,Symptoms_Headache,Symptoms_Mood Swings
13,34,29.57,5.4,34.0,26.0,6.0,30.000000,5.656854,7,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
14,34,29.57,5.4,38.0,34.0,7.0,32.666667,6.110101,8,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
15,34,29.57,5.4,29.0,38.0,4.0,31.750000,5.315073,9,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
16,34,29.57,5.4,34.0,29.0,5.0,32.200000,4.711688,10,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
17,34,29.57,5.4,31.0,34.0,7.0,32.000000,4.242641,11,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


### 4. Split and Shape for Models - Format B (For LSTM Networks)

In [8]:
LOOKBACK_STEPS = 2

def create_sequences_for_lstm(X_data, y_data, user_ids, lookback_steps=LOOKBACK_STEPS):
    X_sequences, y_sequences = [], []
    for user_id in user_ids.unique():
        user_X = X_data[user_ids == user_id].values
        user_y = y_data[user_ids == user_id].values

        for i in range(len(user_X) - lookback_steps):
            # Ensure sequences do NOT cross boundaries between different users.
            # This is implicitly handled by iterating per user.
            X_sequences.append(user_X[i:(i + lookback_steps)])
            y_sequences.append(user_y[i + lookback_steps]) # Predict the next cycle length

    return np.array(X_sequences), np.array(y_sequences)

# Create sequences for LSTM using the same train/test split of users
# Need to use the original `df_final_features` which still contains 'User ID' for this step

# First, re-extract X and y that include User ID for correct grouping
X_lstm_full = df_final_features.drop(columns=['Cycle Length'])
y_lstm_full = df_final_features['Cycle Length']
user_ids_lstm_full = df_final_features['User ID']

# Filter for train and test users as defined before
X_lstm_train = X_lstm_full[user_ids_lstm_full.isin(train_users)].drop(columns=['User ID'])
y_lstm_train = y_lstm_full[user_ids_lstm_full.isin(train_users)]
user_ids_train_lstm = user_ids_lstm_full[user_ids_lstm_full.isin(train_users)]

X_lstm_test = X_lstm_full[user_ids_lstm_full.isin(test_users)].drop(columns=['User ID'])
y_lstm_test = y_lstm_full[user_ids_lstm_full.isin(test_users)]
user_ids_test_lstm = user_ids_lstm_full[user_ids_lstm_full.isin(test_users)]

# Generate sequences
X_train_lstm, y_train_lstm = create_sequences_for_lstm(X_lstm_train, y_lstm_train, user_ids_train_lstm, LOOKBACK_STEPS)
X_test_lstm, y_test_lstm = create_sequences_for_lstm(X_lstm_test, y_lstm_test, user_ids_test_lstm, LOOKBACK_STEPS)

print("Data shaped for LSTM Networks (Format B):")
print(f"X_train_lstm shape: {X_train_lstm.shape}, y_train_lstm shape: {y_train_lstm.shape}")
print(f"X_test_lstm shape: {X_test_lstm.shape}, y_test_lstm shape: {y_test_lstm.shape}")

print("\nExample of an LSTM sequence (first sample from X_train_lstm):")
print(X_train_lstm[0])
print("Corresponding target for this sequence (first sample from y_train_lstm):")
print(y_train_lstm[0])

Data shaped for LSTM Networks (Format B):
X_train_lstm shape: (386, 2, 22), y_train_lstm shape: (386,)
X_test_lstm shape: (109, 2, 22), y_test_lstm shape: (109,)

Example of an LSTM sequence (first sample from X_train_lstm):
[[34.         29.57        5.4        34.         26.          6.
  30.          5.65685425  7.          0.          0.          1.
   0.          1.          0.          0.          1.          0.
   0.          1.          0.          0.        ]
 [34.         29.57        5.4        38.         34.          7.
  32.66666667  6.11010093  8.          0.          0.          1.
   0.          1.          0.          0.          1.          0.
   0.          0.          0.          1.        ]]
Corresponding target for this sequence (first sample from y_train_lstm):
34


### Training XGBoost Model

In [9]:
import xgboost as xgb

# Instantiate XGBoost Regressor
# You can tune hyperparameters as needed. For now, using default.
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', # Objective for regression tasks
    n_estimators=100,            # Number of boosting rounds
    random_state=42
)

# Train the model using the log-transformed target variable
# We need to re-create y_train_log_transformed using the y_train from the split

# First, re-define X and y from the cleaned dataframe based on the training split
X_train_for_xgb = X[user_ids_for_split.isin(train_users)]
y_train_for_xgb = y[user_ids_for_split.isin(train_users)]

# Apply log1p transformation to the training target variable
y_train_log_transformed = np.log1p(y_train_for_xgb)

xgb_model.fit(X_train_for_xgb, y_train_log_transformed)

print("XGBoost model training complete using log-transformed target.")

XGBoost model training complete using log-transformed target.


### Evaluate XGBoost Model Performance

In [10]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Prepare X_test and y_test for evaluation (ensure they are based on the correct split)
# First, re-define X and y from the cleaned dataframe based on the testing split
X_test_for_xgb = X[user_ids_for_split.isin(test_users)]
y_test_for_xgb = y[user_ids_for_split.isin(test_users)]

# Make predictions on the X_test set
y_pred_log_transformed = xgb_model.predict(X_test_for_xgb)

# Inverse transform the predictions to get them back to the original scale
y_pred = np.expm1(y_pred_log_transformed)

# Calculate evaluation metrics
r2 = r2_score(y_test_for_xgb, y_pred)
mae = mean_absolute_error(y_test_for_xgb, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_for_xgb, y_pred))

print(f"XGBoost Model Performance on Test Set:\n")
print(f"R-squared (R²): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

# Display some actual vs predicted values
results_df = pd.DataFrame({'Actual': y_test_for_xgb, 'Predicted': y_pred})
print("\nActual vs Predicted (first 10 values):")
display(results_df.head(10))

XGBoost Model Performance on Test Set:

R-squared (R²): -0.1718
Mean Absolute Error (MAE): 6.9271
Root Mean Squared Error (RMSE): 8.3003

Actual vs Predicted (first 10 values):


,Actual,Predicted
2,41,35.914261
3,27,37.345100
4,42,36.008686
5,41,33.466721
6,31,49.787415
7,48,41.934105
8,29,38.959595
9,47,39.861923
10,42,42.313828
42,38,32.342213


### Hyperparameter Tuning for XGBoost using GridSearchCV

In [11]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300], # Number of boosting rounds
    'learning_rate': [0.01, 0.05, 0.1], # Step size shrinkage to prevent overfitting
    'max_depth': [3, 5, 7],         # Maximum depth of a tree
    'subsample': [0.7, 0.8, 0.9],   # Subsample ratio of the training instance
    'colsample_bytree': [0.7, 0.8, 0.9], # Subsample ratio of columns when constructing each tree
    'gamma': [0, 0.1, 0.2]          # Minimum loss reduction required to make a further partition
}

# Instantiate XGBoost Regressor with fixed parameters
# Use a reasonable random_state for reproducibility
xgb_base_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

# Instantiate GridSearchCV
# We will optimize for negative mean absolute error (MAE) or R-squared.
# Let's use negative MAE as it directly relates to the absolute difference in days.
# n_jobs=-1 uses all available processors.
# cv=3 or cv=5 is common for cross-validation folds.
grid_search = GridSearchCV(
    estimator=xgb_base_model,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error', # Optimize for lower MAE (hence negative)
    cv=3,                            # 3-fold cross-validation
    n_jobs=-1,                       # Use all available CPU cores
    verbose=1                        # Print progress messages
)

# Fit GridSearchCV to the training data (features and log-transformed target)
# X_train_for_xgb and y_train_log_transformed are already defined from previous steps
print("Starting GridSearchCV...")
grid_search.fit(X_train_for_xgb, y_train_log_transformed)

print("GridSearchCV complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best MAE score (negative): {grid_search.best_score_:.4f}")
print(f"Best MAE score (positive): {-grid_search.best_score_:.4f}")

# Store the best estimator
best_xgb_model = grid_search.best_estimator_

print("\nBest XGBoost model parameters have been identified and stored.")

Starting GridSearchCV...
Fitting 3 folds for each of 729 candidates, totalling 2187 fits
GridSearchCV complete.
Best parameters found: {'colsample_bytree': 0.7, 'gamma': 0.2, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.7}
Best MAE score (negative): -0.1692
Best MAE score (positive): 0.1692

Best XGBoost model parameters have been identified and stored.


### Evaluate Tuned XGBoost Model Performance

In [12]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Make predictions on the X_test set using the best_xgb_model
y_pred_tuned_log_transformed = best_xgb_model.predict(X_test_for_xgb)

# Inverse transform the predictions to get them back to the original scale
y_pred_tuned = np.expm1(y_pred_tuned_log_transformed)

# Calculate evaluation metrics for the tuned model
r2_tuned = r2_score(y_test_for_xgb, y_pred_tuned)
mae_tuned = mean_absolute_error(y_test_for_xgb, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test_for_xgb, y_pred_tuned))

print(f"Tuned XGBoost Model Performance on Test Set:\n")
print(f"R-squared (R²): {r2_tuned:.4f}")
print(f"Mean Absolute Error (MAE): {mae_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_tuned:.4f}")

# Display some actual vs predicted values for the tuned model
results_tuned_df = pd.DataFrame({'Actual': y_test_for_xgb, 'Predicted (Tuned)': y_pred_tuned})
print("\nActual vs Predicted (Tuned Model - first 10 values):")
display(results_tuned_df.head(10))

Tuned XGBoost Model Performance on Test Set:

R-squared (R²): 0.0045
Mean Absolute Error (MAE): 6.6540
Root Mean Squared Error (RMSE): 7.6506

Actual vs Predicted (Tuned Model - first 10 values):


,Actual,Predicted (Tuned)
2,41,36.487740
3,27,35.894493
4,42,37.864635
5,41,35.894493
6,31,38.006134
7,48,36.563751
8,29,37.279671
9,47,37.864635
10,42,37.864635
42,38,34.418484


### Training Random Forest Model

In [13]:
from sklearn.ensemble import RandomForestRegressor

# Instantiate RandomForestRegressor
# Using some default parameters for now, can be tuned later if needed
rf_model = RandomForestRegressor(
    n_estimators=100,      # Number of trees in the forest
    random_state=42,       # For reproducibility
    n_jobs=-1              # Use all available processors
)

# Train the model using the same training features and log-transformed target
# X_train_for_xgb and y_train_log_transformed are already defined from previous steps
rf_model.fit(X_train_for_xgb, y_train_log_transformed)

print("Random Forest model training complete using log-transformed target.")

Random Forest model training complete using log-transformed target.


### Evaluate Random Forest Model Performance

In [14]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Make predictions on the X_test set using the rf_model
y_pred_rf_log_transformed = rf_model.predict(X_test_for_xgb)

# Inverse transform the predictions to get them back to the original scale
y_pred_rf = np.expm1(y_pred_rf_log_transformed)

# Calculate evaluation metrics for the Random Forest model
r2_rf = r2_score(y_test_for_xgb, y_pred_rf)
mae_rf = mean_absolute_error(y_test_for_xgb, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_for_xgb, y_pred_rf))

print(f"Random Forest Model Performance on Test Set:\n")
print(f"R-squared (R²): {r2_rf:.4f}")
print(f"Mean Absolute Error (MAE): {mae_rf:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf:.4f}")

# Display some actual vs predicted values for the Random Forest model
results_rf_df = pd.DataFrame({'Actual': y_test_for_xgb, 'Predicted (RF)': y_pred_rf})
print("\nActual vs Predicted (Random Forest Model - first 10 values):")
display(results_rf_df.head(10))

Random Forest Model Performance on Test Set:

R-squared (R²): -0.0154
Mean Absolute Error (MAE): 6.6788
Root Mean Squared Error (RMSE): 7.7266

Actual vs Predicted (Random Forest Model - first 10 values):


,Actual,Predicted (RF)
2,41,37.571278
3,27,37.348722
4,42,37.402347
5,41,35.958310
6,31,36.626670
7,48,35.698865
8,29,38.247581
9,47,36.394310
10,42,38.479002
42,38,33.085177


### Hyperparameter Tuning for Random Forest using GridSearchCV

In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

# Define the parameter grid to search for Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],  # Number of trees in the forest
    'max_depth': [5, 10, 15],         # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],    # Minimum number of samples required to be at a leaf node
    'max_features': ['sqrt', 'log2']  # Number of features to consider when looking for the best split
}

# Instantiate RandomForestRegressor with fixed parameters
rf_base_model = RandomForestRegressor(random_state=42, n_jobs=-1)

# Instantiate GridSearchCV for Random Forest
grid_search_rf = GridSearchCV(
    estimator=rf_base_model,
    param_grid=param_grid_rf,
    scoring='neg_mean_absolute_error', # Optimize for lower MAE (hence negative)
    cv=3,                            # 3-fold cross-validation
    n_jobs=-1,                       # Use all available CPU cores
    verbose=1                        # Print progress messages
)

# Fit GridSearchCV to the training data (features and log-transformed target)
# X_train_for_xgb and y_train_log_transformed are already defined from previous steps
print("Starting GridSearchCV for Random Forest...")
grid_search_rf.fit(X_train_for_xgb, y_train_log_transformed)

print("GridSearchCV for Random Forest complete.")
print(f"Best parameters found for Random Forest: {grid_search_rf.best_params_}")
print(f"Best MAE score (negative) for Random Forest: {grid_search_rf.best_score_:.4f}")
print(f"Best MAE score (positive) for Random Forest: {-grid_search_rf.best_score_:.4f}")

# Store the best estimator
best_rf_model = grid_search_rf.best_estimator_

print("\nBest Random Forest model parameters have been identified and stored.")

Starting GridSearchCV for Random Forest...
Fitting 3 folds for each of 162 candidates, totalling 486 fits
GridSearchCV for Random Forest complete.
Best parameters found for Random Forest: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}
Best MAE score (negative) for Random Forest: -0.1701
Best MAE score (positive) for Random Forest: 0.1701

Best Random Forest model parameters have been identified and stored.


### Evaluate Tuned Random Forest Model Performance

In [16]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Make predictions on the X_test set using the best_rf_model
y_pred_rf_tuned_log_transformed = best_rf_model.predict(X_test_for_xgb)

# Inverse transform the predictions to get them back to the original scale
y_pred_rf_tuned = np.expm1(y_pred_rf_tuned_log_transformed)

# Calculate evaluation metrics for the tuned Random Forest model
r2_rf_tuned = r2_score(y_test_for_xgb, y_pred_rf_tuned)
mae_rf_tuned = mean_absolute_error(y_test_for_xgb, y_pred_rf_tuned)
rmse_rf_tuned = np.sqrt(mean_squared_error(y_test_for_xgb, y_pred_rf_tuned))

print(f"Tuned Random Forest Model Performance on Test Set:\n")
print(f"R-squared (R²): {r2_rf_tuned:.4f}")
print(f"Mean Absolute Error (MAE): {mae_rf_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf_tuned:.4f}")

# Display some actual vs predicted values for the tuned Random Forest model
results_rf_tuned_df = pd.DataFrame({'Actual': y_test_for_xgb, 'Predicted (RF Tuned)': y_pred_rf_tuned})
print("\nActual vs Predicted (Tuned Random Forest Model - first 10 values):")
display(results_rf_tuned_df.head(10))

Tuned Random Forest Model Performance on Test Set:

R-squared (R²): 0.0090
Mean Absolute Error (MAE): 6.6689
Root Mean Squared Error (RMSE): 7.6330

Actual vs Predicted (Tuned Random Forest Model - first 10 values):


,Actual,Predicted (RF Tuned)
2,41,36.974350
3,27,36.590695
4,42,37.295663
5,41,36.869213
6,31,37.903848
7,48,36.745028
8,29,37.059502
9,47,37.410486
10,42,37.462702
42,38,34.733531


In [17]:
pip install scikeras

### Training Artificial Neural Network (ANN) Model with GridSearchCV

In [26]:
import tensorflow as tf
from tensorflow import keras
from scikeras.wrappers import KerasRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Define a function to create the Keras model
def create_model(n_hidden_layers=1, n_neurons=50, learning_rate=0.001, activation='relu'):
    model = keras.models.Sequential()
    # Fix: Wrap X_train_for_xgb.shape[1] in a tuple
    model.add(keras.layers.InputLayer(input_shape=(X_train_for_xgb.shape[1],)))
    for _ in range(n_hidden_layers):
        model.add(keras.layers.Dense(n_neurons, activation=activation))
    model.add(keras.layers.Dense(1)) # Output layer for regression
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='mean_squared_error', optimizer=optimizer)
    return model

print("Training a single ANN model with fixed parameters due to GridSearchCV compatibility issues.")

# Instantiate a KerasRegressor with a reasonable set of parameters
# (e.g., using parameters that might have been found by GridSearchCV or common defaults)
# We'll use 1 hidden layer, 100 neurons, learning rate 0.001, relu activation.
# And train for 50 epochs with a batch size of 32.
best_ann_model = KerasRegressor(
    model=create_model,
    n_hidden_layers=1,
    n_neurons=100,
    learning_rate=0.001,
    activation='relu',
    batch_size=32,
    epochs=50, # Increased epochs for better training than GridSearchCV's limited ones
    verbose=0 # Suppress Keras output during training
)

# Explicitly set _estimator_type for compatibility, though KerasRegressor usually handles this.
# This line is kept for robustness, even if the primary GridSearchCV issue is different.
best_ann_model._estimator_type = "regressor"

# Train the single ANN model
print("Starting ANN model training...")
best_ann_model.fit(X_train_for_xgb, y_train_log_transformed)

print("ANN model training complete.")
print("\nSingle ANN model has been trained and stored as best_ann_model.")

Training a single ANN model with fixed parameters due to GridSearchCV compatibility issues.
Starting ANN model training...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


ANN model training complete.

Single ANN model has been trained and stored as best_ann_model.


### Evaluate Tuned ANN Model Performance

In [28]:
# Make predictions on the X_test set using the best_ann_model
y_pred_ann_tuned_log_transformed = best_ann_model.predict(X_test_for_xgb)

# Inverse transform the predictions to get them back to the original scale
y_pred_ann_tuned = np.expm1(y_pred_ann_tuned_log_transformed)

# Calculate evaluation metrics for the tuned ANN model
r2_ann_tuned = r2_score(y_test_for_xgb, y_pred_ann_tuned)
mae_ann_tuned = mean_absolute_error(y_test_for_xgb, y_pred_ann_tuned)
rmse_ann_tuned = np.sqrt(mean_squared_error(y_test_for_xgb, y_pred_ann_tuned))

print(f"Tuned ANN Model Performance on Test Set:\n")
print(f"R-squared (R²): {r2_ann_tuned:.4f}")
print(f"Mean Absolute Error (MAE): {mae_ann_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ann_tuned:.4f}")

# Display some actual vs predicted values for the tuned ANN model
results_ann_tuned_df = pd.DataFrame({'Actual': y_test_for_xgb, 'Predicted (ANN Tuned)': y_pred_ann_tuned.flatten()})
print("\nActual vs Predicted (Tuned ANN Model - first 10 values):")
display(results_ann_tuned_df.head(10))

Tuned ANN Model Performance on Test Set:

R-squared (R²): -1.7769
Mean Absolute Error (MAE): 10.2527
Root Mean Squared Error (RMSE): 12.7775

Actual vs Predicted (Tuned ANN Model - first 10 values):


,Actual,Predicted (ANN Tuned)
2,41,17.839357
3,27,22.333614
4,42,17.433849
5,41,18.418835
6,31,33.601486
7,48,29.076689
8,29,22.232721
9,47,18.684395
10,42,24.234367
42,38,45.218765


### Evaluate Tuned ANN Model Performance

In [27]:
# Make predictions on the X_test set using the best_ann_model
y_pred_ann_tuned_log_transformed = best_ann_model.predict(X_test_for_xgb)

# Inverse transform the predictions to get them back to the original scale
y_pred_ann_tuned = np.expm1(y_pred_ann_tuned_log_transformed)

# Calculate evaluation metrics for the tuned ANN model
r2_ann_tuned = r2_score(y_test_for_xgb, y_pred_ann_tuned)
mae_ann_tuned = mean_absolute_error(y_test_for_xgb, y_pred_ann_tuned)
rmse_ann_tuned = np.sqrt(mean_squared_error(y_test_for_xgb, y_pred_ann_tuned))

print(f"Tuned ANN Model Performance on Test Set:\n")
print(f"R-squared (R²): {r2_ann_tuned:.4f}")
print(f"Mean Absolute Error (MAE): {mae_ann_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ann_tuned:.4f}")

# Display some actual vs predicted values for the tuned ANN model
results_ann_tuned_df = pd.DataFrame({'Actual': y_test_for_xgb, 'Predicted (ANN Tuned)': y_pred_ann_tuned.flatten()})
print("\nActual vs Predicted (Tuned ANN Model - first 10 values):")
display(results_ann_tuned_df.head(10))

Tuned ANN Model Performance on Test Set:

R-squared (R²): -1.7769
Mean Absolute Error (MAE): 10.2527
Root Mean Squared Error (RMSE): 12.7775

Actual vs Predicted (Tuned ANN Model - first 10 values):


,Actual,Predicted (ANN Tuned)
2,41,17.839357
3,27,22.333614
4,42,17.433849
5,41,18.418835
6,31,33.601486
7,48,29.076689
8,29,22.232721
9,47,18.684395
10,42,24.234367
42,38,45.218765


### Compare All Model Performances

### Training Artificial Neural Network (LSTM) Model

In [30]:
import tensorflow as tf
from tensorflow import keras
from scikeras.wrappers import KerasRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Define a function to create the LSTM model
def create_lstm_model(n_lstm_layers=1, n_neurons=50, learning_rate=0.001, activation='relu', dropout_rate=0.2):
    model = keras.models.Sequential()
    # Input shape for LSTM layers is (timesteps, features)
    # X_train_lstm.shape[1] is timesteps (LOOKBACK_STEPS), X_train_lstm.shape[2] is features
    model.add(keras.layers.LSTM(n_neurons, activation=activation, return_sequences=True if n_lstm_layers > 1 else False, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
    model.add(keras.layers.Dropout(dropout_rate))

    for _ in range(n_lstm_layers - 1):
        model.add(keras.layers.LSTM(n_neurons, activation=activation, return_sequences=True if _ < n_lstm_layers - 2 else False))
        model.add(keras.layers.Dropout(dropout_rate))

    model.add(keras.layers.Dense(1)) # Output layer for regression
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='mean_squared_error', optimizer=optimizer)
    return model

print("Training a single LSTM model with fixed parameters due to GridSearchCV compatibility issues.")

# Instantiate a KerasRegressor with a reasonable set of parameters
# (e.g., using common defaults and what worked for ANN)
# We'll use 1 LSTM layer, 100 neurons, learning rate 0.001, relu activation, dropout 0.2.
# And train for 50 epochs with a batch size of 32.
best_lstm_model = KerasRegressor(
    model=create_lstm_model,
    n_lstm_layers=1,
    n_neurons=100,
    learning_rate=0.001,
    activation='relu',
    dropout_rate=0.2,
    batch_size=32,
    epochs=50,
    verbose=0 # Suppress Keras output during training
)

# Explicitly set _estimator_type for compatibility
best_lstm_model._estimator_type = "regressor"

# Train the single LSTM model
print("Starting LSTM model training...")
best_lstm_model.fit(X_train_lstm, np.log1p(y_train_lstm))

print("LSTM model training complete.")
print("\nSingle LSTM model has been trained and stored as best_lstm_model.")

Training a single LSTM model with fixed parameters due to GridSearchCV compatibility issues.
Starting LSTM model training...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


LSTM model training complete.

Single LSTM model has been trained and stored as best_lstm_model.


### Evaluate LSTM Model Performance

In [31]:
# Make predictions on the X_test_lstm set using the best_lstm_model
y_pred_lstm_log_transformed = best_lstm_model.predict(X_test_lstm)

# Inverse transform the predictions to get them back to the original scale
y_pred_lstm = np.expm1(y_pred_lstm_log_transformed)

# Calculate evaluation metrics for the LSTM model
r2_lstm = r2_score(y_test_lstm, y_pred_lstm)
mae_lstm = mean_absolute_error(y_test_lstm, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test_lstm, y_pred_lstm))

print(f"LSTM Model Performance on Test Set:\n")
print(f"R-squared (R²): {r2_lstm:.4f}")
print(f"Mean Absolute Error (MAE): {mae_lstm:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_lstm:.4f}")

# Display some actual vs predicted values for the LSTM model
results_lstm_df = pd.DataFrame({'Actual': y_test_lstm, 'Predicted (LSTM)': y_pred_lstm.flatten()})
print("\nActual vs Predicted (LSTM Model - first 10 values):")
display(results_lstm_df.head(10))

LSTM Model Performance on Test Set:

R-squared (R²): -1.6908
Mean Absolute Error (MAE): 10.2798
Root Mean Squared Error (RMSE): 12.4286

Actual vs Predicted (LSTM Model - first 10 values):


,Actual,Predicted (LSTM)
0,42,36.955956
1,41,34.133907
2,31,32.720524
3,48,33.784111
4,29,34.453968
5,47,27.511274
6,42,40.958260
7,30,29.866657
8,36,26.947739
9,37,29.161150


### Compare All Model Performances

In [32]:
print("Comparison of Tuned Model Performances on Test Set:\n")

print(f"Tuned XGBoost Model:     R²={r2_tuned:.4f}, MAE={mae_tuned:.4f}, RMSE={rmse_tuned:.4f}")
print(f"Tuned Random Forest Model: R²={r2_rf_tuned:.4f}, MAE={mae_rf_tuned:.4f}, RMSE={rmse_rf_tuned:.4f}")
print(f"Tuned ANN Model:         R²={r2_ann_tuned:.4f}, MAE={mae_ann_tuned:.4f}, RMSE={rmse_ann_tuned:.4f}")
print(f"LSTM Model:              R²={r2_lstm:.4f}, MAE={mae_lstm:.4f}, RMSE={rmse_lstm:.4f}")

print("\nBased on these results, we can see which model performs best across the chosen metrics.")
print("It's important to note that if all models still show very poor performance (R² near zero),")
print("it might indicate limitations in the current feature set, data quality, or the inherent predictability of the problem.")

Comparison of Tuned Model Performances on Test Set:

Tuned XGBoost Model:     R²=0.0045, MAE=6.6540, RMSE=7.6506
Tuned Random Forest Model: R²=0.0090, MAE=6.6689, RMSE=7.6330
Tuned ANN Model:         R²=-1.7769, MAE=10.2527, RMSE=12.7775
LSTM Model:              R²=-1.6908, MAE=10.2798, RMSE=12.4286

Based on these results, we can see which model performs best across the chosen metrics.
It's important to note that if all models still show very poor performance (R² near zero),
it might indicate limitations in the current feature set, data quality, or the inherent predictability of the problem.


## Interactive Prediction Interface

Use the form below to input the necessary details for a new cycle, and the trained models will predict the next cycle length.

In [36]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np

# Define the input widgets for each feature

# Numerical features
age_widget = widgets.IntSlider(min=18, max=60, value=30, description='Age:')
bmi_widget = widgets.FloatSlider(min=15.0, max=40.0, step=0.1, value=25.0, description='BMI:')
sleep_hours_widget = widgets.FloatSlider(min=4.0, max=12.0, step=0.5, value=7.0, description='Sleep Hours:')

# NEW: Textarea for historical cycle lengths
recent_cycle_lengths_widget = widgets.Textarea(
    value='30,32,28',  # Example: last 3 cycle lengths
    description='Recent Cycle Lengths (comma-separated):',
    layout={'width': 'auto', 'height': 'auto'}
)

# NEW: Input for Current Cycle Start Date
current_cycle_start_date_widget = widgets.Text(value='2023-01-01', description='Current Cycle Start Date (YYYY-MM-DD):')

# start_month is still a direct input, representing the start month of the *new* cycle being predicted.
# This will now be derived from current_cycle_start_date_widget for consistency.
# For now, keep it as a slider for flexibility, but note the intent to derive it.
start_month_widget = widgets.IntSlider(min=1, max=12, value=1, description='Start Month (of the *new* cycle):')

# Categorical features (matching the one-hot encoding columns)
stress_level_options = ['1', '2', '3', '4', '5']
stress_level_widget = widgets.Dropdown(options=stress_level_options, value='3', description='Stress Level:')

exercise_frequency_options = ['Low', 'Moderate', 'High']
exercise_frequency_widget = widgets.Dropdown(options=exercise_frequency_options, value='Moderate', description='Exercise Frequency:')

diet_options = ['Vegan', 'Vegetarian', 'Non-Vegetarian']
diet_widget = widgets.Dropdown(options=diet_options, value='Non-Vegetarian', description='Diet:')

symptoms_options = ['None', 'Acne', 'Bloating', 'Cramps', 'Fatigue', 'Headache', 'Mood Swings']
symptoms_widget = widgets.Dropdown(options=symptoms_options, value='None', description='Symptoms:')

predict_button = widgets.Button(description='Predict Cycle Length and Next Start Date')
output_area = widgets.Output()

# Global means/medians for NaN filling from training data (assuming X_train_for_xgb is available from previous cells)
global_mean_pcl1 = X_train_for_xgb['prev_cycle_len_1'].mean()
global_mean_pcl2 = X_train_for_xgb['prev_cycle_len_2'].mean()
global_mean_urm = X_train_for_xgb['user_running_mean'].mean()
global_mean_urs = X_train_for_xgb['user_running_std'].mean()


# Function to prepare input data for prediction
def prepare_input_data(age, bmi, sleep_hours, recent_cycles_str, sm, sl, ef, dt, sym):
    # Parse recent cycle lengths
    try:
        recent_cycles = [int(c.strip()) for c in recent_cycles_str.split(',') if c.strip()]
        if not recent_cycles:
            raise ValueError("No recent cycle lengths provided.")
    except ValueError:
        with output_area:
            print("Please enter valid comma-separated numbers for 'Recent Cycle Lengths'.")
        return None # Indicate failure

    # Calculate lagged features and running statistics from the recent_cycles list
    num_recent_cycles = len(recent_cycles)

    # prev_cycle_len_1: the most recent cycle length
    pcl1 = recent_cycles[-1] if num_recent_cycles >= 1 else global_mean_pcl1
    # prev_cycle_len_2: the second most recent cycle length
    pcl2 = recent_cycles[-2] if num_recent_cycles >= 2 else global_mean_pcl2

    # user_running_mean and user_running_std are calculated from *all* provided recent cycles
    if num_recent_cycles >= 1:
        urm = np.mean(recent_cycles)
        urs = np.std(recent_cycles)
    else:
        urm = global_mean_urm
        urs = global_mean_urs

    # Create a DataFrame with the input features
    input_df = pd.DataFrame({
        'Age': [age],
        'BMI': [bmi],
        'Sleep Hours': [sleep_hours],
        'prev_cycle_len_1': [pcl1],
        'prev_cycle_len_2': [pcl2],
        'user_running_mean': [urm],
        'user_running_std': [urs],
        'start_month': [sm],
        'Stress Level': [sl],
        'Exercise Frequency': [ef],
        'Diet': [dt],
        'Symptoms': [sym]
    })

    # Apply one-hot encoding (matching training data columns)
    expected_columns = X_train_for_xgb.columns.tolist()

    categorical_cols_to_process = ['Stress Level', 'Exercise Frequency', 'Diet', 'Symptoms']
    processed_df = pd.get_dummies(input_df, columns=categorical_cols_to_process, drop_first=True, dtype=float)

    # Ensure all expected columns are present, fill missing with 0
    for col in expected_columns:
        if col not in processed_df.columns:
            processed_df[col] = 0.0

    # Ensure column order matches training data
    processed_df = processed_df[expected_columns]

    return processed_df

# Prediction function
def on_predict_button_clicked(b):
    with output_area:
        clear_output()
        print("Generating predictions...")

        current_cycle_start_date_str = current_cycle_start_date_widget.value
        try:
            current_cycle_start_date = pd.to_datetime(current_cycle_start_date_str)
            # Update start_month_widget value from the current_cycle_start_date for consistency
            # This makes the start_month widget reflective of the entered date.
            start_month_widget.value = current_cycle_start_date.month
        except ValueError:
            print(f"Error: Invalid date format for 'Current Cycle Start Date'. Please use YYYY-MM-DD. Got: {current_cycle_start_date_str}")
            return

        input_for_models = prepare_input_data(
            age_widget.value,
            bmi_widget.value,
            sleep_hours_widget.value,
            recent_cycle_lengths_widget.value,
            start_month_widget.value, # Use the potentially updated start_month
            stress_level_widget.value,
            exercise_frequency_widget.value,
            diet_widget.value,
            symptoms_widget.value
        )

        if input_for_models is None: # Error in input parsing
            return

        # Prepare input for LSTM (requires sequence of previous steps)
        lstm_single_input_features = input_for_models.values.flatten()
        input_for_lstm = np.array([lstm_single_input_features] * LOOKBACK_STEPS).reshape(1, LOOKBACK_STEPS, -1)

        # Make predictions
        # XGBoost
        y_pred_xgb_log = best_xgb_model.predict(input_for_models)
        y_pred_xgb = np.expm1(y_pred_xgb_log)

        # Random Forest
        y_pred_rf_log = best_rf_model.predict(input_for_models)
        y_pred_rf = np.expm1(y_pred_rf_log)

        # ANN
        y_pred_ann_log = best_ann_model.predict(input_for_models)
        y_pred_ann = np.expm1(y_pred_ann_log)

        # LSTM
        y_pred_lstm_log = best_lstm_model.predict(input_for_lstm)
        y_pred_lstm = np.expm1(y_pred_lstm_log)

        print("\n--- Predictions ---")

        # Calculate Next Cycle Start Date for each model
        next_cycle_date_xgb = current_cycle_start_date + pd.to_timedelta(round(y_pred_xgb[0]), unit='D')
        next_cycle_date_rf = current_cycle_start_date + pd.to_timedelta(round(y_pred_rf[0]), unit='D')
        next_cycle_date_ann = current_cycle_start_date + pd.to_timedelta(round(y_pred_ann[0]), unit='D') # Corrected
        next_cycle_date_lstm = current_cycle_start_date + pd.to_timedelta(round(y_pred_lstm[0]), unit='D')

        print(f"XGBoost Prediction: {y_pred_xgb[0]:.2f} days (Next Cycle Start Date: {next_cycle_date_xgb.strftime('%Y-%m-%d')})")
        print(f"Random Forest Prediction: {y_pred_rf[0]:.2f} days (Next Cycle Start Date: {next_cycle_date_rf.strftime('%Y-%m-%d')})")
        print(f"ANN Prediction: {y_pred_ann[0]:.2f} days (Next Cycle Start Date: {next_cycle_date_ann.strftime('%Y-%m-%d')})") # Corrected
        print(f"LSTM Prediction: {y_pred_lstm[0]:.2f} days (Next Cycle Start Date: {next_cycle_date_lstm.strftime('%Y-%m-%d')})")


# Attach the click event to the button
predict_button.on_click(on_predict_button_clicked)

# Arrange widgets in a VBox
input_widgets = widgets.VBox([
    age_widget,
    bmi_widget,
    sleep_hours_widget,
    recent_cycle_lengths_widget,
    current_cycle_start_date_widget, # New widget
    start_month_widget,
    stress_level_widget,
    exercise_frequency_widget,
    diet_widget,
    symptoms_widget,
    predict_button,
    output_area
])

display(input_widgets)